# RNNModel_Word_Prediction_Project

1. Project Objective
2. Import Libraries
3. Training Text / Dataset
4. Tokenization
5. Create Input-Output Sequences
6. Padding
7. Build RNN Model
8. Compile
9. Train
10. Test Next-Word Prediction
11. Custom User Input
12. Final Prediction

**Project Goal: Given a sequence of words, predict only the next word.**
**bold text**

**Input:      i love this
Prediction: movie**

**# RNN Model — Next Word Prediction

## Objective

The goal of this project is to build a Recurrent Neural Network (RNN)
that predicts the next word from a given sequence of words.

### Example

Input:
    "i love this"

Output:
    "movie"

The model learns word-to-word relationships from the training text.**

# 16 — Final Pipeline



# RNN Word Prediction Pipeline

Raw Text
   ↓
Tokenization
   ↓
Word IDs
   ↓
Create N-gram Sequences
   ↓
Padding
   ↓
X = Previous Words
y = Next Word
   ↓
Embedding
   ↓
SimpleRNN
   ↓
Dense + Softmax
   ↓
Word Probabilities
   ↓
argmax()
   ↓
Predicted Next Word

In [1]:
# 1  Import Libraries
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


# simplernn->sequence ko process karke learn karta hai
# dense->gen prob

In [2]:
# 2  training dataset

text = """
i love this movie
i love this content
i love this story
i love this song
this movie is very good
this movie is very interesting
this content is very good
this story is very interesting
i really love this movie
i really love this content
"""



# Model isi text se seekhega ki kaunse words ek doosre ke baad commonly aate hain.
# For example:
# i → love
# i love → this
# i love this → movie

In [3]:
# 4 — Tokenization

# words ko numbers (IDs)
tokenizer = tf.keras.preprocessing.text.Tokenizer()

tokenizer.fit_on_texts([text])

# vocabulary size count karta hai  ye len  and
total_words = len(tokenizer.word_index) + 1

print("Vocabulary Size:", total_words)
print("Word Index:")
# {"i": 1, "love": 2, "this": 3, "movie": 4}  mapping akrte hai
print(tokenizer.word_index)


Vocabulary Size: 13
Word Index:
{'this': 1, 'i': 2, 'love': 3, 'movie': 4, 'is': 5, 'very': 6, 'content': 7, 'story': 8, 'good': 9, 'interesting': 10, 'really': 11, 'song': 12}


# VVI CELL OF THIS PROJECT

In [4]:
# 5 — Create Input-Output Sequences

# vvi cell

input_sequences = []

for line in text.strip().split("\n"):

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print("Total sequences:", len(input_sequences))
print(input_sequences[:10])




# ?s happening here
# sentence=["i love this movie"]
# Token IDs maan lo:
# i = 1   , love = 3 ,this = 2 ,movie = 5

# toh sequences banengi: and Isi se model ko input-output relationship milta hai.
# [1, 3]           i              → love
# [1, 3, 2]       i love         → this
# [1, 3, 2, 5]     i love this    → movie




Total sequences: 36
[[2, 3], [2, 3, 1], [2, 3, 1, 4], [2, 3], [2, 3, 1], [2, 3, 1, 7], [2, 3], [2, 3, 1], [2, 3, 1, 8], [2, 3]]


**padding**
** **
Different sequences ki length different hai, isliye padding karni padegi.
** **

In [5]:
max_sequence_len = max([len(seq) for seq in input_sequences])

input_sequences = np.array(
    tf.keras.preprocessing.sequence.pad_sequences(
        input_sequences,
        maxlen=max_sequence_len,
        padding="pre"
    )
)

print("Maximum Sequence Length:", max_sequence_len)
print(input_sequences[:10])
# sentence  ki max len   5 hamen rkahe bcz jitne v hamare   datasewrt men sentencse hain bow 4 len ki hin hai so padding   5 karde  taki jismen len  akm v ho bow 0 add hoke 5 ki ho jaye
# Before:          After:
# [1, 3]           [0, 1, 3, 2, 5]
# [1, 3, 2]        [0, 0, 1, 3, 2]
# [1, 3, 2, 5]     [0, 1, 3, 2, 5]




Maximum Sequence Length: 5
[[0 0 0 2 3]
 [0 0 2 3 1]
 [0 2 3 1 4]
 [0 0 0 2 3]
 [0 0 2 3 1]
 [0 2 3 1 7]
 [0 0 0 2 3]
 [0 0 2 3 1]
 [0 2 3 1 8]
 [0 0 0 2 3]]


# 7 — X and y Separate Karna
**   **
brakdown  into
X = previous words (input),
y = next word (target/output)
**   **

In [6]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)



# explanation
# suppose padded sequence after pading
#     [0, 1, 3, 2, 5]
# X = [0, 1, 3, 2] y=5  Matlab:
# X = previous words    y = next word
#     i love this     →     movie

# X = i love this
# y = movie

X shape: (36, 4)
y shape: (36,)


# 8 — Build RNN Model

In [7]:
model = Sequential([

    # Word IDs → word vectors
    Embedding(
        input_dim=total_words,
        output_dim=64
    ),

    # Sequence processing  RNN sequence ko process karke learn karta hai
    # "i love this" → "movie"
    # RNN layer mein 128 units/neurons hain jo sequence ka context learn aur process karte hain
    SimpleRNN(128),

    # Probability of every word in vocabulary
    Dense(
        total_words,
        activation="softmax"
    )
])

model.summary()


# imp yahan
# Dense(total_words, activation="softmax")
# isliye hai kyunki model ko vocabulary ke har possible word
# ki probability calculate karni hai.
# eg:  > Highest probability: movie as of prob cal;cvulated

# movie      → 0.72
# content    → 0.15
# story      → 0.06
# song       → 0.03
# ...
# Architecture
# i/p                "i love this"
# Input Word IDs     [1, 3, 2]
#       ↓
# Embedding          ahr word   64 numbers ka vector
#       ↓
# SimpleRNN       128
#       ↓
# Dense            Har possible word ki probability
#       ↓
# Softmax
#       ↓
# Probability of every word

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# 9 — Compile

In [8]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Why this loss?
# y mein actual next word integer ID hai.
# Example:movie → 5
# Isliye:  sparse_categorical_crossentropy  use kar rahe hain.

# 10 — Train Model

In [ ]:
history = model.fit(
    X,
    y,
    epochs=200,
    verbose=1
)



# Training:

# X (i/p data)
# ↓
# Embedding (convert token--> numerical vector)
# ↓
# RNN
# ↓
# Dense + Softmax
# ↓
# Prediction
# ↓
# Loss
# ↓
# Backpropagation
# ↓
# Weights Update

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.1667 - loss: 2.5238
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.2778 - loss: 2.4237
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3056 - loss: 2.3330
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3056 - loss: 2.2471
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3056 - loss: 2.1523
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3056 - loss: 2.0660
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3056 - loss: 1.9866
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3056 - loss: 1.9269
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.3056 - loss: 1.8699
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3333 - loss: 1.8209
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4167 - loss: 1.7540
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.4444 - lo

# 11 — Create Reverse Word Index

In [ ]:
reverse_word_index = {
    value: key
    for key, value in tokenizer.word_index.items()
}

print(reverse_word_index)




# Model output mein word nahi, word ID milegi.
# Isliye ID → word conversion ke liye:
# eg: 5 → movie
#     8 → content

# 12 — vvi    Next Word Prediction Function

In [ ]:
def predict_next_word(seed_text):
# seed_text = "i love"


    # Text → Token IDs
    # ["i love"]  --->  [1 , 2]
    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    # Same sequence length as training
    token_list = tf.keras.preprocessing.sequence.pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,
        padding="pre"
        # [1, 2]
        #    ↓
        # [0, 0, 1, 2]   pre eman s  0  ko pahle  add kiya hai
    )

    # Predict probabilities  of each  words
    prediction = model.predict(token_list, verbose=0)

    # Highest probability word ID   ke  id hin iss predicted_id=3  uyaise store hua
    predicted_id = np.argmax(prediction, axis=-1)[0]

    # Word ID → Actual word    3 → "this"
    predicted_word = reverse_word_index.get(
        predicted_id,
        ""
    )

    return predicted_word




#   Input
# "i love"
#    ↓
# tokenizer
#    ↓
# [1, 2]
#    ↓
# padding
#    ↓
# [0, 0, 1, 2]
#    ↓
# model.predict()
#    ↓
# probabilities
#    ↓
# np.argmax()
#    ↓
# predicted_id = 3
#    ↓
# reverse_word_index
#    ↓
# "this"
#    ↓
# return
# finalyy   "i love" → next predicted word "this"

# 13 — Test the Model

In [ ]:
test_sentence = "i love this"

next_word = predict_next_word(test_sentence)

print("Input:", test_sentence)
print("Predicted Next Word:", next_word)






# Expected output Because training data contains:i love this movie
# you should ideally get:
# Input: i love this
# Predicted Next Word: movie

# Depending on training and the tiny dataset, output can vary.

**14 — Test Multiple Inputs**

In [ ]:
test_sentences = [
    "i love this",
    "i love this movie",
    "this movie is",
    "this content is",
    "i really love this"
]

for sentence in test_sentences:

    next_word = predict_next_word(sentence)

    print(
        sentence,
        "→",
        next_word
    )



#     Expected type of output
# i love this → movie
# i love this movie → ...
# this movie is → very
# this content is → very
# i really love this → movie

**15 — User Input Demo**

In [ ]:
while True:

    user_input = input(
        "\nEnter a sentence (type 'exit' to stop): "
    )

    if user_input.lower() == "exit":
        break

    next_word = predict_next_word(user_input)

    print("Predicted Next Word:", next_word)




    # i/p->Enter a sentence: i love this
    # o/p->Predicted Next Word:    movie
    # i/p->Enter a sentence: this movie is
    # o/p->Predicted Next Word:     very


# Conclusion

The RNN model successfully learns sequential relationships
between words and predicts the most probable next word.

For example:

Input:
    "i love this"

Prediction:
    "movie"

The project demonstrates how RNNs can be used for
basic language modeling and next-word prediction.